# 주의사항
<br> 1. 실행 전 사용할 수 있는 gpu 확인하기
<br> 2. scene 몇개로 설정한것인지 정하기
<br> 3. 결과를 저장하는 log path 이름 잘 바꾸기

# MSCP1
<br>Multi-modal Sensing-aided Channel Prediction for 6G mmWave Massive Antenna Systems
<br>위 논문을 참고해서 CPPN과 VCRN 구현후 모델 예측을 진행
<br>논문의 경우 과거 채널 n개를 입력으로 미래 채널 n개 만큼의 채널 예측 파라미터를 추출(CPPN)
<br>그 후 예측 파라미터와 image를 fusion해서 채널 예측을 진행함(VCRN)
<br>MSCP = CPPN + VCRN

# import

In [1]:
import os
import sys

import numpy as  np
from VCRN import VCRN
from CPPN import CPPN
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset as TorchDataset, DataLoader, Subset
import torch.nn as nn   
import time



from PIL import Image
import numpy as np

from deepverse import ParameterManager
from deepverse.scenario import ScenarioManager
from deepverse import Dataset

from deepverse.visualizers import ImageVisualizer, LidarVisualizer

# Setting
## CUDA 몇번 사용하는지 잘 확인하기

In [2]:
# Scenes 1000
## Subcarriers 64

scenarios_name = "DT31"
config_path = f"scenarios/{scenarios_name}/param/config.m"
param_manager = ParameterManager(config_path)

params = param_manager.get_params()

param_manager.params["scenes"] =list(range(1000))
param_manager.params["comm"]["OFDM"]["selected_subcarriers"] = list(range(64))

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"현재 사용 중인 장치: {device}")


현재 사용 중인 장치: cuda:0


# Generate a dataset

In [3]:
# scenes / subcarriers는 그대로
param_manager.params["radar"]["enable"] = False

# lidar가 dict로 존재하면 disable
if isinstance(param_manager.params.get("lidar", None), dict):
    param_manager.params["lidar"]["enable"] = False

# position이 dict로 존재하면 disable
if isinstance(param_manager.params.get("position", None), dict):
    param_manager.params["position"]["enable"] = False

# Radar, LiDAR, Position 사용하지 않으므로 False 
dataset = Dataset(param_manager)

Generating camera dataset: ⏳ In progress
Generating camera dataset: ✅ Completed (0.00s)
Generating LiDAR dataset: ⏳ In progress
Generating LiDAR dataset: ✅ Completed (0.00s)
Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (9.14s)


# Processing
### Channel

In [4]:
def get_coeffs_from_frame(frame, ue_idx=0):
    ue_obj = frame["ue"]

    # 케이스1) list/tuple이면 ue_idx로 선택
    if isinstance(ue_obj, (list, tuple)):
        ch_obj = ue_obj[ue_idx]
    else:
        # 케이스2) 단일 OFDMChannel이면 그대로 사용
        ch_obj = ue_obj

    # coeffs는 dict key가 아니라 attribute일 확률이 매우 큼
    if hasattr(ch_obj, "coeffs"):
        return ch_obj.coeffs

    # 혹시 dict라면 마지막 보험
    if isinstance(ch_obj, dict) and "coeffs" in ch_obj:
        return ch_obj["coeffs"]

    raise TypeError(f"Cannot get coeffs. ue type={type(ue_obj)}, ch type={type(ch_obj)}")


In [5]:
def get_train_min_max_realimag(frames, train_idx, us_idx=0):

    rmin, rmax =  float('inf'), float('-inf')
    imin, imax =  float('inf'), float('-inf')

    print("Calculating min/max over training set...")

    for t  in train_idx:
        frame  = frames[t]
        cooeffs  = get_coeffs_from_frame(frame, us_idx)  # (N_subcarriers, )

        rmin = min(rmin, float(cooeffs.real.min()))
        rmax = max(rmax, float(cooeffs.real.max()))
        imin = min(imin, float(cooeffs.imag.min()))
        imax = max(imax, float(cooeffs.imag.max()))

    print(f"Done. rmin={rmin}, rmax={rmax}, imin={imin}, imax={imax}")
    return (rmin, rmax), (imin, imax)

In [6]:
def preprocess_channel_coeffs_minmax(coeffs_np, r_min, r_max, i_min, i_max, device=device, eps=1e-16):
    # Convert Numpy to Tensor
    coeffs = torch.from_numpy(coeffs_np).to(torch.complex64)
    
    
    r = coeffs.real
    i = coeffs.imag
    
    # Min-Max Scaling [0, 1]
    # Add eps to denominator to prevent division by zero
    r_scaled = (r - r_min) / max(r_max - r_min, eps)
    i_scaled = (i - i_min) / max(i_max - i_min, eps)
    
    # Concat (Maintains shape like (..., 2*subcarriers))
    H = torch.cat([r_scaled, i_scaled], dim=-1).to(dtype=torch.float32)
    return H

### image

In [7]:
IMG_SIZE = 224
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32).view(1,3,1,1)

def preprocess_img(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("RGB")
    arr = np.array(img)

    x = torch.from_numpy(arr).permute(2, 0, 1).contiguous().float()
    x = x / 255.0
    x = x.unsqueeze(0)

    x = F.interpolate(x, size=(img_size, img_size), mode="bilinear", align_corners=False)

    # mean/std는 전역으로 빼는 게 성능상 좋음 (아래 참고)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    return x.squeeze(0)  # (3,224,224) CPU


# Dataset 구현

In [8]:
def flatten_comm_frames(comm):
    frames = []
    for row in comm.data:
        for d in row:
            frames.append(d)
    return frames

class MultiModalNextStepDatasetGPU(TorchDataset):
    def __init__(self, comm_frames, cam_files, ue_idx=0, past_len=15, device=device,
                 # Arguments for statistical values (initialized with default values)
                 r_min=0.0, r_max=1.0, i_min=0.0, i_max=1.0):
        
        self.comm_frames = comm_frames
        self.cam_files = list(cam_files)
        self.ue_idx = ue_idx
        self.past_len = past_len
        self.device = device
        
        # Save statistical values
        self.r_min, self.r_max = r_min, r_max
        self.i_min, self.i_max = i_min, i_max

        self.N = min(len(self.comm_frames), len(self.cam_files))
        self.valid_start = past_len - 1
        self.valid_end = self.N - 2 

    def __len__(self):
        return self.valid_end - self.valid_start + 1

    def __getitem__(self, idx):
        t = self.valid_start + idx

        # 1. Image Past (Apply Preprocessing)
        img_list = []
        for k in range(t - self.past_len + 1, t + 1):
            img_path = self.cam_files[k]
            img_k = preprocess_img(img_path).squeeze(0)
            img_list.append(img_k)
        img = torch.stack(img_list, dim=0)  # Shape: (past_len

        
        # 2. Channel Past (Apply Scaling)
        ch_list = []
        for k in range(t - self.past_len + 1, t + 1):
            coeffs_np = get_coeffs_from_frame(self.comm_frames[k], ue_idx=self.ue_idx)
            # Use the newly defined Min-Max preprocessing function
            h = preprocess_channel_coeffs_minmax(
                coeffs_np, 
                self.r_min, self.r_max, self.i_min, self.i_max, 
                device=self.device
            ).reshape(-1)
            ch_list.append(h)
        channel_past = torch.stack(ch_list, dim=0)

        # 3. Target (Apply Scaling) - Target must also be scaled for model training!
        coeffs_np_next = get_coeffs_from_frame(self.comm_frames[t + 1], ue_idx=self.ue_idx)
        target = preprocess_channel_coeffs_minmax(
            coeffs_np_next, 
            self.r_min, self.r_max, self.i_min, self.i_max, 
            device=self.device
        ).reshape(-1)

        return channel_past, img, target

# DataLoader 구현

In [9]:
comm_frames = flatten_comm_frames(dataset.comm_dataset)
sensor = dataset.camera_dataset.sensors["unit1_cam1"]

ds = MultiModalNextStepDatasetGPU(
    comm_frames=comm_frames,
    cam_files=sensor.files,
    ue_idx=0,
    past_len=16,
    device=device
)

loader = DataLoader(
    ds,
    batch_size=8,
    shuffle=True,
    num_workers=8,     
    pin_memory=True   
)

ch, img, y = next(iter(loader))
print(ch.shape, img.shape, y.shape)
print(ch.device, img.device, y.device)


torch.Size([8, 16, 2048]) torch.Size([8, 16, 3, 224, 224]) torch.Size([8, 2048])
cpu cpu cpu


# Fine-tuning

In [ ]:
import torch
import torch.nn as nn

class MSCP_Predictor(nn.Module):
    """
    VCRN(backbone) = CPPN + (image_embedding + fusion_layers + ffn)

    Input:
      ch:  (B, T, F_in)           e.g. (B,16,2048)
      img: (B, T, 3,224,224) or (B,3,224,224)
    Output:
      yhat:(B, F_out)            e.g. (B,2048)
    """
    def __init__(
        self,
        backbone: nn.Module,      # VCRN() instance
        F_in: int,
        F_out: int,
        element_length: int = 64, # CPPN input token dim
        d_model: int = 128,       # token dim
        pool: str = "last",       # "last" or "mean"
        use_last_image_only: bool = True,  # img가 5D면 last frame만 사용
        freeze_cppn: bool = False,
        freeze_image: bool = False,
        freeze_fusion: bool = False,
    ):
        super().__init__()
        self.backbone = backbone
        self.pool = pool
        self.use_last_image_only = use_last_image_only

        # (B,T,F_in) -> (B,T,64)
        self.in_proj = nn.Sequential(
            nn.Linear(F_in, 512),
            nn.GELU(),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Linear(256, element_length),
        )

        # (B,128) -> (B,F_out)
        self.head = nn.Linear(d_model, F_out)

        # ---- freezing options ----
        if freeze_cppn and hasattr(self.backbone, "cppn"):
            for p in self.backbone.cppn.parameters():
                p.requires_grad = False

        if freeze_image and hasattr(self.backbone, "image_embedding"):
            for p in self.backbone.image_embedding.parameters():
                p.requires_grad = False

        if freeze_fusion:
            # fusion_layers + ffn freeze
            if hasattr(self.backbone, "fusion_layers"):
                for p in self.backbone.fusion_layers.parameters():
                    p.requires_grad = False
            if hasattr(self.backbone, "ffn"):
                for p in self.backbone.ffn.parameters():
                    p.requires_grad = False

    def forward(self, ch, img):
        # ch: (B,T,F_in) -> (B,T,64)
        ch = self.in_proj(ch)

        # img: (B,T,3,224,224)면 처음엔 last frame만 쓰는게 안정적
        if img.dim() == 5 and self.use_last_image_only:
            img_in = img[:, -1]  # (B,3,224,224)
        else:
            img_in = img         # (B,3,224,224) or (B,T,3,224,224)

        # backbone(VCRN): (B,T,64), img -> (B,T,128)
        tokens = self.backbone(ch, img_in)

        # pooling
        if self.pool == "last":
            z = tokens[:, -1, :]      # (B,128)
        elif self.pool == "mean":
            z = tokens.mean(dim=1)    # (B,128)
        else:
            raise ValueError(f"Unknown pool={self.pool}")

        return self.head(z)           # (B,F_out)

# NMSE(dB)

In [11]:
@torch.no_grad()
def nmse_db(yhat: torch.Tensor, y: torch.Tensor, eps: float = 1e-16) -> torch.Tensor:
    # yhat, y: (B,F)
    num = torch.sum((yhat - y) ** 2, dim=1)
    den = torch.sum(y ** 2, dim=1).clamp_min(eps)
    nmse = num / den
    return 10.0 * torch.log10(nmse.clamp_min(eps)).mean()


# Train/Val Split

In [12]:
# 겹치는 부분이 있는지 확인하는 함수
def used_frames_for_ds_indices(ds, ds_indices):
    used = set()
    for i in ds_indices:
        t = ds.valid_start + i
        # inputs use [t-T+1 .. t], target uses (t+1)
        used.update(range(t - T + 1, t + 2))
    return used

In [13]:
n_total = len(ds)
T = ds.past_len  # = past_len

# ---- choose k so that train=3k, val=k and NO frame-overlap is possible ----
k_max = (n_total - T) // 4
if k_max <= 0:
    raise ValueError(
        f"Not enough data for strict non-overlap 3:1. "
        f"n_total={n_total}, past_len(T)={T}. "
        f"Try increasing scenes or decreasing past_len."
    )

k = k_max               # use as much data as possible under constraints
n_val = k
n_train = 3 * k

train_idx = list(range(0, n_train))
val_idx   = list(range(n_total - n_val, n_total))

print("n_total:", n_total, "T:", T)
print("train samples:", len(train_idx), "val samples:", len(val_idx), "ratio:", len(train_idx)/len(val_idx))

# ---- min/max ONLY from train (same as your original policy) ----
train_ts = [ds.valid_start + i for i in train_idx]

(real_min, real_max), (imag_min, imag_max) = get_train_min_max_realimag(
    comm_frames, train_ts, us_idx=0
)

ds.r_min = real_min
ds.r_max = real_max
ds.i_min = imag_min
ds.i_max = imag_max

print("Dataset statistical values set in the dataset.")

train_ds = Subset(ds, train_idx)
val_ds   = Subset(ds, val_idx)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=8)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=8)

F_in = 2048


overlap = used_frames_for_ds_indices(ds, train_idx).intersection(
          used_frames_for_ds_indices(ds, val_idx))
print("frame-overlap count:", len(overlap))
assert len(overlap) == 0, "Leakage detected: train/val share frames!"

# Verify
ch, img, y = next(iter(train_loader))
F_out = y.shape[-1]
print("\n=== Data Check ===")
print(f"y stats | min: {y.min().item():.4f}, max: {y.max().item():.4f}")
print("If scaling worked correctly, values should be within [0, 1].")

n_total: 984 T: 16
train samples: 726 val samples: 242 ratio: 3.0
Calculating min/max over training set...
Done. rmin=-1.599962900128591e-06, rmax=1.5977824378041814e-06, imin=-1.6400732538319291e-06, imax=1.579536450012739e-06
Dataset statistical values set in the dataset.
frame-overlap count: 0

=== Data Check ===
y stats | min: 0.1765, max: 0.8418
If scaling worked correctly, values should be within [0, 1].


# Train/Eval 함수

In [14]:
def train_one_epoch(model, loader, optimizer, device, grad_clip=1.0):
    model.train()

    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, img, y in loader:
        # Dataset이 이미 cuda 텐서를 반환하더라도 안전하게 유지
        ch = ch.to(device, non_blocking=True)
        img = img.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        yhat = model(ch, img)
        loss = F.mse_loss(yhat, y)
        
        loss.backward()

        if grad_clip is not None and grad_clip > 0:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()
        
        total_loss += loss.item()
        total_nmse += nmse_db(yhat.detach(), y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()

    total_loss = 0.0
    total_nmse = 0.0
    n = 0

    for ch, img, y in loader:
        ch = ch.to(device, non_blocking=True)
        img = img.to(device, non_blocking=True)
        y  = y.to(device, non_blocking=True)

        yhat = model(ch, img)
        loss = F.mse_loss(yhat, y)

        total_loss += loss.item()
        total_nmse += nmse_db(yhat, y).item()
        n += 1

    return total_loss / max(n, 1), total_nmse / max(n, 1)


# 모델 생성 및 확인

In [15]:
# 배치 하나로 F_in/F_out 자동 확정
ch, img, y = next(iter(train_loader))
F_in  = ch.shape[-1]
F_out = y.shape[-1]
print("Detected:", "F_in=", F_in, "F_out=", F_out)
print("Batch devices:", ch.device, img.device, y.device)

# backbone + finetune model
backbone = VCRN().to(device)

model = MSCP_Predictor(
    backbone=backbone,
    F_in=F_in,
    F_out=F_out,
    element_length=64,
    d_model=128,
    pool="last",
    use_last_image_only=True,   # ✅ 지금 VCRN 5D 처리 애매하면 True 유지 권장
    freeze_cppn=False,
    freeze_image=False,
    freeze_fusion=False,
).to(device)

# sanity forward
model.eval()
with torch.no_grad():
    # ds가 이미 cuda 텐서 반환이면 아래 .to(device) 생략 가능
    yhat = model(ch.to(device), img.to(device))
print("yhat:", yhat.shape, "y:", y.shape)



Detected: F_in= 2048 F_out= 2048
Batch devices: cpu cpu cpu
yhat: torch.Size([32, 2048]) y: torch.Size([32, 2048])


# Optimizer/Scheduler 설정

In [16]:
# requires_grad=True인 파라미터만 학습
trainable_params = [p for p in model.parameters() if p.requires_grad]
print("trainable params:", sum(p.numel() for p in trainable_params))

optimizer = torch.optim.AdamW(trainable_params, lr=1e-4, weight_decay=1e-4)

# (선택) cosine scheduler
epochs = 300
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


trainable params: 6635456


# 학습 루프 및 checkpoint 저장

In [17]:
import os
import time
import torch

best_val = float("inf")
best_val_nmse = None
best_epoch = None

t_train0 = time.time()

log_path = "MSCP1_scene1000.txt"  # ✅ 파일명만 바꿔서 관리

with open(log_path, "w", encoding="utf-8") as f:
    # ✅ (선택) 실험 메타정보 헤더
    header = (
        "=== Experiment Info ===\n"
        f"model=LWM_multi_finetune\n"
        f"scenes=1000\n"
        f"epochs={epochs}\n"
        f"F_in={F_in}, F_out={F_out}\n"
        f"start_time={time.strftime('%Y-%m-%d %H:%M:%S')}\n"
        "=======================\n"
    )
    print(header)
    f.write(header + "\n")
    f.flush()

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        tr_loss, tr_nmse = train_one_epoch(model, train_loader, optimizer, device=device, grad_clip=1.0)
        va_loss, va_nmse = evaluate(model, val_loader, device=device)

        scheduler.step()

        dt = time.time() - t0
        line = (
            f"[{epoch:02d}/{epochs}] "
            f"train loss={tr_loss:.6f}, nmse(dB)={tr_nmse:.4f} | "
            f"val loss={va_loss:.6f}, nmse(dB)={va_nmse:.4f} | "
            f"{dt:.1f}s"
        )

        print(line)
        f.write(line + "\n")
        f.flush()

        if va_loss < best_val:
            best_val = va_loss
            best_val_nmse = va_nmse
            best_epoch = epoch

            ckpt_path = "best_MSCP1_finetune.pt"
            torch.save(
                {
                    "epoch": epoch,
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "F_in": F_in,
                    "F_out": F_out,
                    "best_val_loss": best_val,
                    "best_val_nmse_db": best_val_nmse,
                },
                ckpt_path
            )

            save_line = (
                f"  ↳ saved {ckpt_path} | best@{best_epoch}: "
                f"val loss={best_val:.6f}, val nmse(dB)={best_val_nmse:.6f}"
            )
            print(save_line)
            f.write(save_line + "\n")
            f.flush()

total_sec = time.time() - t_train0
h = int(total_sec // 3600)
m = int((total_sec % 3600) // 60)
s = total_sec % 60

summary1 = "\n=== Training Summary ==="
summary2 = f"Total time: {total_sec:.1f}s ({h}h {m}m {s:.1f}s)"
summary3 = f"Best epoch: {best_epoch} | best val loss={best_val:.6f}, best val nmse(dB)={best_val_nmse:.6f}"

print(summary1); print(summary2); print(summary3)

with open(log_path, "a", encoding="utf-8") as f:
    f.write(summary1 + "\n")
    f.write(summary2 + "\n")
    f.write(summary3 + "\n")

print("Log saved to:", os.path.abspath(log_path))

=== Experiment Info ===
model=LWM_multi_finetune
scenes=1000
epochs=300
F_in=2048, F_out=2048
start_time=2026-03-04 23:55:41

[01/300] train loss=0.395510, nmse(dB)=1.5540 | val loss=0.266237, nmse(dB)=-0.0813 | 132.4s
  ↳ saved best_MSCP1_finetune.pt | best@1: val loss=0.266237, val nmse(dB)=-0.081283
[02/300] train loss=0.202765, nmse(dB)=-1.3451 | val loss=0.130438, nmse(dB)=-3.2120 | 132.0s
  ↳ saved best_MSCP1_finetune.pt | best@2: val loss=0.130438, val nmse(dB)=-3.211957
[03/300] train loss=0.101253, nmse(dB)=-4.3984 | val loss=0.064016, nmse(dB)=-6.4141 | 131.3s
  ↳ saved best_MSCP1_finetune.pt | best@3: val loss=0.064016, val nmse(dB)=-6.414142
[04/300] train loss=0.054288, nmse(dB)=-7.1730 | val loss=0.035973, nmse(dB)=-9.1968 | 131.8s
  ↳ saved best_MSCP1_finetune.pt | best@4: val loss=0.035973, val nmse(dB)=-9.196798
[05/300] train loss=0.034430, nmse(dB)=-9.2802 | val loss=0.025116, nmse(dB)=-11.1936 | 130.9s
  ↳ saved best_MSCP1_finetune.pt | best@5: val loss=0.025116, va

# 런타임 체크

In [ ]:
ch, img, y = next(iter(train_loader))
print("y abs mean:", y.abs().mean().item())
print("y abs max :", y.abs().max().item())
print("y power   :", (y**2).mean().item())

with torch.no_grad():
    yhat = model(ch.to(device), img.to(device))
print("yhat abs mean:", yhat.abs().mean().item())
print("yhat abs max :", yhat.abs().max().item())
print("yhat power   :", (yhat**2).mean().item())


y abs mean: 0.506757378578186
y abs max : 0.892146646976471
y power   : 0.27473005652427673
yhat abs mean: 0.5043463110923767
yhat abs max : 0.5205907821655273
yhat power   : 0.2543974220752716


In [30]:
y[0:15, : 15]

tensor([[0.5877, 0.5991, 0.6083, 0.6154, 0.6201, 0.6228, 0.6238, 0.6234, 0.6222,
         0.6204, 0.6184, 0.6163, 0.6143, 0.6120, 0.6092],
        [0.4409, 0.4536, 0.4663, 0.4788, 0.4910, 0.5030, 0.5146, 0.5258, 0.5365,
         0.5467, 0.5565, 0.5659, 0.5749, 0.5834, 0.5917],
        [0.5474, 0.5596, 0.5720, 0.5837, 0.5948, 0.6065, 0.6198, 0.6340, 0.6469,
         0.6566, 0.6628, 0.6672, 0.6720, 0.6779, 0.6841],
        [0.7429, 0.7453, 0.7477, 0.7512, 0.7563, 0.7631, 0.7712, 0.7798, 0.7881,
         0.7955, 0.8022, 0.8084, 0.8146, 0.8209, 0.8272],
        [0.3976, 0.3864, 0.3792, 0.3747, 0.3721, 0.3710, 0.3723, 0.3765, 0.3831,
         0.3901, 0.3958, 0.3996, 0.4031, 0.4082, 0.4162],
        [0.4742, 0.4856, 0.4971, 0.5088, 0.5204, 0.5320, 0.5434, 0.5545, 0.5650,
         0.5749, 0.5839, 0.5921, 0.5993, 0.6057, 0.6113],
        [0.5747, 0.5600, 0.5453, 0.5322, 0.5217, 0.5140, 0.5085, 0.5039, 0.4989,
         0.4923, 0.4836, 0.4731, 0.4618, 0.4505, 0.4402],
        [0.5283, 0.5397, 0.

In [ ]:
yhat[:15, :15]

tensor([[0.5067, 0.5071, 0.5071, 0.5084, 0.5078, 0.5076, 0.5073, 0.5070, 0.5072,
         0.5082, 0.5079, 0.5073, 0.5085, 0.5076, 0.5089],
        [0.5078, 0.5076, 0.5077, 0.5085, 0.5065, 0.5074, 0.5070, 0.5073, 0.5080,
         0.5091, 0.5071, 0.5078, 0.5090, 0.5089, 0.5089],
        [0.5076, 0.5076, 0.5069, 0.5080, 0.5072, 0.5066, 0.5087, 0.5064, 0.5070,
         0.5085, 0.5077, 0.5068, 0.5081, 0.5073, 0.5089],
        [0.5060, 0.5084, 0.5075, 0.5086, 0.5074, 0.5076, 0.5072, 0.5065, 0.5065,
         0.5092, 0.5075, 0.5079, 0.5087, 0.5071, 0.5094],
        [0.5048, 0.5097, 0.5084, 0.5073, 0.5070, 0.5073, 0.5047, 0.5070, 0.5062,
         0.5085, 0.5062, 0.5088, 0.5094, 0.5060, 0.5092],
        [0.5081, 0.5068, 0.5062, 0.5082, 0.5082, 0.5073, 0.5080, 0.5061, 0.5076,
         0.5080, 0.5091, 0.5071, 0.5075, 0.5076, 0.5085],
        [0.5072, 0.5071, 0.5072, 0.5082, 0.5081, 0.5074, 0.5065, 0.5067, 0.5067,
         0.5083, 0.5075, 0.5069, 0.5080, 0.5080, 0.5087],
        [0.5074, 0.5077, 0.

: 

# 데이터 입력 형태

In [19]:
cam_files=sensor.files
print("=== dataset sizes ===")
print("N(comm_frames):", len(comm_frames))
print("N(cam_files)  :", len(cam_files))
print("N(min)        :", min(len(comm_frames), len(cam_files)))
print("past_len      :", ds.past_len)
print("len(ds)       :", len(ds))
print("len(train_ds) :", len(train_ds))
print("len(val_ds)   :", len(val_ds))
print("len(train_loader):", len(train_loader))
print("len(val_loader)  :", len(val_loader))

print("\n=== one batch shapes ===")
ch, img, y = next(iter(train_loader))
print("ch :", tuple(ch.shape), " -> (B,T,F_in)")
print("img:", tuple(img.shape), " -> (B,3,224,224)")
print("y  :", tuple(y.shape), " -> (B,F_out)")
with torch.no_grad():
    yhat = model(ch.to(device), img.to(device))
print("yhat:", tuple(yhat.shape), " -> (B,F_out)")
print("this forward predicted vectors:", yhat.shape[0], "(=B)")
print("each vector predicts elements:", yhat.shape[1], "(=F_out)")


=== dataset sizes ===
N(comm_frames): 1000
N(cam_files)  : 7012
N(min)        : 1000
past_len      : 16
len(ds)       : 984
len(train_ds) : 726
len(val_ds)   : 242
len(train_loader): 23
len(val_loader)  : 8

=== one batch shapes ===
ch : (32, 16, 2048)  -> (B,T,F_in)
img: (32, 16, 3, 224, 224)  -> (B,3,224,224)
y  : (32, 2048)  -> (B,F_out)
yhat: (32, 2048)  -> (B,F_out)
this forward predicted vectors: 32 (=B)
each vector predicts elements: 2048 (=F_out)
